# Notebook 02 — Final RibFrac 500-Patient Preprocessing and Patient Registry

This notebook is the **final preprocessing/registry stage** for the RibFrac labelled development cohort.

## Source cohort
- Training Part 1: 300 CTs
- Training Part 2: 120 CTs
- Tuning/Validation: 80 CTs
- Total labelled patients: **500**

## Final patient-level protocol
The three labelled sources are combined first and then split at the **patient level**:
- Training: 350 patients
- Validation: 75 patients
- Internal test: 75 patients

The internal test set is distinct from the official RibFrac challenge test set, whose ground truth is not publicly available.

## Important storage design
The full 1-mm isotropic preprocessed volumes are **not permanently materialized for all 500 patients** in `/kaggle/working`. Instead, this notebook:
1. validates all source metadata;
2. creates and saves the final patient registry/split;
3. provides a reproducible, on-demand patient preprocessing function;
4. optionally materializes selected patients for smoke testing.

Notebook 03 can use the same on-demand preprocessing logic to extract patches patient-by-patient, avoiding a 100+ GB intermediate volume store.


In [ ]:
# Cell 1 — Imports
import gc
import hashlib
import io
import json
import zlib
import tempfile
import re
import os
import random
import struct
import time
import zipfile
from pathlib import Path
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
import nibabel as nib
from scipy.ndimage import zoom
from sklearn.model_selection import train_test_split

print("Libraries imported successfully.")


In [ ]:
# Cell 2 — Reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("Random seed:", SEED)


In [ ]:
# Cell 3 — Final source configuration

# Official Zenodo records
SOURCES = {
    "train_part1": {
        "record": "3893508",
        "expected_patients": 300,
        "info_filename": "ribfrac-train-info-1.csv",
        "image_zip": "ribfrac-train-images-1.zip",
        "label_zip": "ribfrac-train-labels-1.zip",
        "image_url": "https://zenodo.org/records/3893508/files/ribfrac-train-images-1.zip",
        "label_url": "https://zenodo.org/records/3893508/files/ribfrac-train-labels-1.zip",
        "info_url": "https://zenodo.org/records/3893508/files/ribfrac-train-info-1.csv",
    },
    "train_part2": {
        "record": "3893498",
        "expected_patients": 120,
        "info_filename": "ribfrac-train-info-2.csv",
        "image_zip": "ribfrac-train-images-2.zip",
        "label_zip": "ribfrac-train-labels-2.zip",
        "image_url": "https://zenodo.org/records/3893498/files/ribfrac-train-images-2.zip",
        "label_url": "https://zenodo.org/records/3893498/files/ribfrac-train-labels-2.zip",
        "info_url": "https://zenodo.org/records/3893498/files/ribfrac-train-info-2.csv",
    },
    "validation": {
        "record": "3893496",
        "expected_patients": 80,
        "info_filename": "ribfrac-val-info.csv",
        "image_zip": "ribfrac-val-images.zip",
        "label_zip": "ribfrac-val-labels.zip",
        "image_url": "https://zenodo.org/records/3893496/files/ribfrac-val-images.zip",
        "label_url": "https://zenodo.org/records/3893496/files/ribfrac-val-labels.zip",
        "info_url": "https://zenodo.org/records/3893496/files/ribfrac-val-info.csv",
    },
}

TOTAL_EXPECTED = sum(v["expected_patients"] for v in SOURCES.values())

print("Expected labelled patients:", TOTAL_EXPECTED)
for name, cfg in SOURCES.items():
    print(f"{name}: {cfg['expected_patients']} patients")


In [ ]:
# Cell 4 — Persistent working/output directories

WORK_ROOT = Path("/kaggle/working/ribfrac_preprocessed_final")
IMAGE_ROOT = WORK_ROOT / "images"
LABEL_ROOT = WORK_ROOT / "labels"
METADATA_ROOT = WORK_ROOT / "metadata"
CACHE_ROOT = WORK_ROOT / "cache"
SMOKE_ROOT = WORK_ROOT / "smoke_test"

for path in [WORK_ROOT, IMAGE_ROOT, LABEL_ROOT, METADATA_ROOT, CACHE_ROOT, SMOKE_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

# Process one or a small number of patients per run while validating the pipeline.
# Set to None only when the available persistent storage is sufficient.
MAX_PATIENTS_PER_RUN = 1
RESUME_EXISTING = True
RETRY_FAILED = False

print("Persistent root:", WORK_ROOT)
print("Images:", IMAGE_ROOT)
print("Labels:", LABEL_ROOT)
print("Metadata:", METADATA_ROOT)
print("MAX_PATIENTS_PER_RUN:", MAX_PATIENTS_PER_RUN)
print("RESUME_EXISTING:", RESUME_EXISTING)


## Source access strategy

Zenodo's image archives are large (for example, Part 1 is 36.6 GB and Part 2 is 14.6 GB), so the notebook does **not** download an entire image archive. citeturn0view0turn1search0

Instead, image archives are accessed using HTTP byte ranges and only the requested patient's ZIP member is downloaded. The small annotation archives are cached locally.


In [ ]:
# Cell 5 — HTTP helpers

def http_get_bytes(url, start=None, end=None, timeout=120):
    headers = {}
    if start is not None and end is not None:
        headers["Range"] = f"bytes={start}-{end}"

    request = Request(
        url,
        headers={
            "User-Agent": "Mozilla/5.0 RibFrac research preprocessing",
            **headers,
        },
    )

    with urlopen(request, timeout=timeout) as response:
        return response.read(), response.headers


def get_remote_size(url):
    request = Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 RibFrac research preprocessing"},
        method="HEAD",
    )
    try:
        with urlopen(request, timeout=120) as response:
            length = response.headers.get("Content-Length")
            if length:
                return int(length)
    except Exception:
        pass

    # Fallback: request one byte and inspect Content-Range.
    request = Request(
        url,
        headers={
            "User-Agent": "Mozilla/5.0 RibFrac research preprocessing",
            "Range": "bytes=0-0",
        },
    )
    with urlopen(request, timeout=120) as response:
        content_range = response.headers.get("Content-Range", "")
        match = re.search(r"/(\d+)$", content_range)
        if match:
            return int(match.group(1))

    raise RuntimeError(f"Could not determine remote size: {url}")


print("HTTP helpers defined.")


In [ ]:
# Cell 6 — ZIP central-directory inspection

def parse_zip64_extra(extra):
    # ZIP64 extra field ID = 0x0001
    pos = 0
    values = []
    while pos + 4 <= len(extra):
        header_id, size = struct.unpack("<HH", extra[pos:pos+4])
        data = extra[pos+4:pos+4+size]
        pos += 4 + size
        if header_id == 0x0001:
            values.append(data)
    return values


def read_zip64_central_directory(url, chunk_size=1024 * 1024):
    size = get_remote_size(url)

    tail_size = min(size, 1024 * 1024)
    tail, _ = http_get_bytes(url, size - tail_size, size - 1)

    # EOCD signature
    eocd_sig = b"PK\x05\x06"
    eocd_pos = tail.rfind(eocd_sig)

    if eocd_pos < 0:
        raise RuntimeError("ZIP End Of Central Directory not found.")

    eocd = tail[eocd_pos:eocd_pos + 22]
    _, disk, cd_disk, entries_disk, entries_total, cd_size32, cd_offset32, comment_len = struct.unpack(
        "<4sHHHHIIH", eocd
    )

    if cd_size32 != 0xFFFFFFFF and cd_offset32 != 0xFFFFFFFF:
        cd_size = cd_size32
        cd_offset = cd_offset32
        entries_total = entries_total
    else:
        # ZIP64 EOCD locator
        locator_sig = b"PK\x06\x07"
        locator_pos = tail.rfind(locator_sig, 0, eocd_pos)
        if locator_pos < 0:
            raise RuntimeError("ZIP64 EOCD locator not found.")

        _, _, zip64_eocd_offset, _ = struct.unpack(
            "<4sIQI", tail[locator_pos:locator_pos + 20]
        )

        read_size = 1024 * 1024
        z64, _ = http_get_bytes(
            url,
            zip64_eocd_offset,
            min(size - 1, zip64_eocd_offset + read_size - 1),
        )

        if z64[:4] != b"PK\x06\x06":
            raise RuntimeError("ZIP64 EOCD record not found.")

        entries_total = struct.unpack_from("<Q", z64, 32)[0]
        cd_size = struct.unpack_from("<Q", z64, 40)[0]
        cd_offset = struct.unpack_from("<Q", z64, 48)[0]

    cd_bytes, _ = http_get_bytes(
        url,
        cd_offset,
        cd_offset + cd_size - 1,
    )

    entries = {}
    pos = 0

    for _ in range(entries_total):
        if cd_bytes[pos:pos+4] != b"PK\x01\x02":
            raise RuntimeError("Invalid central-directory signature.")

        fixed = cd_bytes[pos:pos+46]
        fields = struct.unpack("<4s6H3I5H2I", fixed)

        (
            signature,
            version_made,
            version_needed,
            flags,
            compression,
            mod_time,
            mod_date,
            crc32,
            comp_size32,
            uncomp_size32,
            name_len,
            extra_len,
            comment_len,
            disk_start,
            int_attr,
            ext_attr,
            local_offset32,
        ) = fields

        name_start = pos + 46
        name = cd_bytes[name_start:name_start + name_len].decode(
            "utf-8", errors="replace"
        )

        extra_start = name_start + name_len
        extra = cd_bytes[extra_start:extra_start + extra_len]

        comp_size = comp_size32
        uncomp_size = uncomp_size32
        local_offset = local_offset32

        if (
            comp_size32 == 0xFFFFFFFF
            or uncomp_size32 == 0xFFFFFFFF
            or local_offset32 == 0xFFFFFFFF
        ):
            zip64_chunks = parse_zip64_extra(extra)
            if not zip64_chunks:
                raise RuntimeError(f"ZIP64 extra field missing for {name}")
            data = zip64_chunks[0]
            zpos = 0
            if uncomp_size32 == 0xFFFFFFFF:
                uncomp_size = struct.unpack_from("<Q", data, zpos)[0]
                zpos += 8
            if comp_size32 == 0xFFFFFFFF:
                comp_size = struct.unpack_from("<Q", data, zpos)[0]
                zpos += 8
            if local_offset32 == 0xFFFFFFFF:
                local_offset = struct.unpack_from("<Q", data, zpos)[0]
                zpos += 8

        entries[name] = {
            "compression": compression,
            "compressed_size": int(comp_size),
            "uncompressed_size": int(uncomp_size),
            "local_header_offset": int(local_offset),
        }

        pos = extra_start + extra_len + comment_len

    return size, entries


print("Remote ZIP inspection helper defined.")


In [ ]:
# Cell 7 — Patient member discovery

def normalize_member_name(name):
    return name.lstrip("/").replace("\\", "/")


def find_patient_member(entries, patient_id, kind):
    patient_id = str(patient_id)

    candidates = []
    for name in entries:
        base = Path(normalize_member_name(name)).name
        if patient_id in base:
            lower = base.lower()
            if kind == "image" and "image" in lower:
                candidates.append(name)
            elif kind == "label" and "label" in lower:
                candidates.append(name)

    if not candidates:
        raise FileNotFoundError(
            f"No {kind} ZIP member found for patient {patient_id}."
        )

    if len(candidates) > 1:
        # Prefer exact conventional filename.
        exact = [
            n for n in candidates
            if Path(n).name == f"{patient_id}-image.nii.gz"
            or Path(n).name == f"{patient_id}-label.nii.gz"
        ]
        if len(exact) == 1:
            return exact[0]

    return sorted(candidates)[0]


def get_zip_member_bytes(url, entry, chunk_size=1024 * 1024):
    # Read local file header to determine variable header length.
    offset = entry["local_header_offset"]

    header, _ = http_get_bytes(
        url,
        offset,
        offset + 30 - 1,
    )

    if header[:4] != b"PK\x03\x04":
        raise RuntimeError("Invalid local ZIP header.")

    (
        signature,
        version,
        flags,
        compression,
        mod_time,
        mod_date,
        crc32,
        comp_size32,
        uncomp_size32,
        name_len,
        extra_len,
    ) = struct.unpack("<4s5H3I2H", header)

    data_start = offset + 30 + name_len + extra_len
    comp_size = entry["compressed_size"]
    data_end = data_start + comp_size - 1

    chunks = []
    for start in range(data_start, data_end + 1, chunk_size):
        end = min(start + chunk_size - 1, data_end)
        chunk, _ = http_get_bytes(url, start, end)
        chunks.append(chunk)

    compressed = b"".join(chunks)

    if compression == 0:
        return compressed

    if compression == 8:
        import zlib
        return zlib.decompress(compressed, -15)

    raise RuntimeError(
        f"Unsupported ZIP compression method: {compression}"
    )


print("Patient ZIP-member download helper defined.")


In [ ]:
# Cell 8 — Download/cache small annotation ZIP archives

def cache_small_file(url, destination):
    destination = Path(destination)
    if destination.exists():
        return destination

    data, _ = http_get_bytes(url)
    destination.write_bytes(data)
    return destination


def load_label_zip_entries(source_name):
    cfg = SOURCES[source_name]
    cache_path = CACHE_ROOT / cfg["label_zip"]

    cache_small_file(cfg["label_url"], cache_path)

    with zipfile.ZipFile(cache_path, "r") as zf:
        return zf.namelist()


for source_name in ["train_part1", "train_part2", "validation"]:
    print(source_name, "label archive:", SOURCES[source_name]["label_zip"])

print("Small annotation archives will be cached on demand.")


In [ ]:
# Cell 9 — Download source metadata

def load_source_metadata(source_name):
    cfg = SOURCES[source_name]
    cache_path = CACHE_ROOT / cfg["info_filename"]

    if not cache_path.exists():
        data, _ = http_get_bytes(cfg["info_url"])
        cache_path.write_bytes(data)

    df = pd.read_csv(cache_path)

    required = {"public_id", "label_id", "label_code"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"{source_name}: metadata missing columns {missing}"
        )

    df["public_id"] = df["public_id"].astype(str)
    df["label_id"] = df["label_id"].astype(int)
    df["label_code"] = df["label_code"].astype(int)
    df["source"] = source_name

    return df


source_metadata = {}

for source_name in SOURCES:
    source_metadata[source_name] = load_source_metadata(source_name)
    df = source_metadata[source_name]

    print(
        source_name,
        "| rows =", len(df),
        "| unique patients =", df["public_id"].nunique()
    )


In [ ]:
# Cell 10 — Verify the three source cohorts

for source_name, cfg in SOURCES.items():
    df = source_metadata[source_name]

    unique_patients = df["public_id"].nunique()

    assert unique_patients == cfg["expected_patients"], (
        f"{source_name}: expected {cfg['expected_patients']} patients, "
        f"found {unique_patients}"
    )

    assert df["public_id"].notna().all()
    assert df["label_id"].notna().all()
    assert df["label_code"].notna().all()

print("All three source metadata counts: PASS")


In [ ]:
# Cell 11 — Verify patient uniqueness across the three sources

patient_sets = {
    source_name: set(df["public_id"].unique())
    for source_name, df in source_metadata.items()
}

source_names = list(patient_sets)

for i in range(len(source_names)):
    for j in range(i + 1, len(source_names)):
        a = source_names[i]
        b = source_names[j]
        overlap = patient_sets[a] & patient_sets[b]

        print(f"{a} ∩ {b}:", len(overlap))

        assert not overlap, (
            f"Patient overlap detected between {a} and {b}: "
            f"{sorted(overlap)[:10]}"
        )

combined_patient_ids = set().union(*patient_sets.values())

assert len(combined_patient_ids) == TOTAL_EXPECTED

print("Unique labelled patients:", len(combined_patient_ids))
print("Cross-source patient uniqueness: PASS")


In [ ]:
# Cell 12 — Combine metadata

combined_metadata = pd.concat(
    source_metadata.values(),
    ignore_index=True
)

combined_metadata = combined_metadata[
    ["public_id", "label_id", "label_code", "source"]
].copy()

combined_metadata = combined_metadata.sort_values(
    ["public_id", "label_id"]
).reset_index(drop=True)

COMBINED_METADATA_PATH = METADATA_ROOT / "combined_ribfrac_metadata.csv"
combined_metadata.to_csv(COMBINED_METADATA_PATH, index=False)

print("Combined metadata:", combined_metadata.shape)
print("Saved:", COMBINED_METADATA_PATH)


## Final patient-level split

The split is created **after combining all 500 labelled cases**. This prevents the Part-1-only split used during development from being accidentally reused.

The split is:
- 70% training = 350 patients
- 15% validation = 75 patients
- 15% internal test = 75 patients

A fixed seed of 42 is used for reproducibility.


In [ ]:
# Cell 13 — Create final 350/75/75 patient split

all_patients = np.array(sorted(combined_patient_ids))

train_patients, temp_patients = train_test_split(
    all_patients,
    test_size=150,
    random_state=SEED,
    shuffle=True,
)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=75,
    random_state=SEED,
    shuffle=True,
)

assert len(train_patients) == 350
assert len(val_patients) == 75
assert len(test_patients) == 75

assert set(train_patients).isdisjoint(val_patients)
assert set(train_patients).isdisjoint(test_patients)
assert set(val_patients).isdisjoint(test_patients)

patient_split = pd.DataFrame({
    "public_id": (
        list(train_patients)
        + list(val_patients)
        + list(test_patients)
    ),
    "split": (
        ["train"] * len(train_patients)
        + ["validation"] * len(val_patients)
        + ["test"] * len(test_patients)
    ),
})

patient_split = patient_split.sort_values("public_id").reset_index(drop=True)

SPLIT_PATH = METADATA_ROOT / "patient_split_500.csv"
patient_split.to_csv(SPLIT_PATH, index=False)

print(patient_split["split"].value_counts())
print("Saved:", SPLIT_PATH)
print("Patient-level split: PASS")


In [ ]:
# Cell 14 — Create patient source registry

source_lookup = (
    combined_metadata[["public_id", "source"]]
    .drop_duplicates()
)

assert not source_lookup["public_id"].duplicated().any()

patient_registry = (
    patient_split
    .merge(source_lookup, on="public_id", how="left", validate="one_to_one")
)

assert patient_registry["source"].notna().all()
assert len(patient_registry) == 500

REGISTRY_PATH = METADATA_ROOT / "patient_source_registry_500.csv"
patient_registry.to_csv(REGISTRY_PATH, index=False)

print(patient_registry.groupby(["source", "split"]).size())
print("Saved:", REGISTRY_PATH)


In [ ]:
# Cell 15 — Verify final source/split counts

summary = (
    patient_registry
    .groupby(["source", "split"])
    .size()
    .unstack(fill_value=0)
)

print(summary)

print("\nTotal by source:")
print(patient_registry["source"].value_counts())

print("\nTotal by split:")
print(patient_registry["split"].value_counts())

assert patient_registry["public_id"].nunique() == 500
assert patient_registry["split"].value_counts().to_dict() == {
    "train": 350,
    "validation": 75,
    "test": 75,
}

print("\nFinal 500-patient registry validation: PASS")


## Preprocessing configuration

The validated development preprocessing is retained:

- Target spacing: `(1.0, 1.0, 1.0)` mm
- CT interpolation: linear/trilinear
- Label interpolation: nearest-neighbor
- HU clipping: `[-1000, 400]`
- CT normalization: `[0, 1]`
- CT dtype: `float32`
- Label dtype: `int16`

Labels remain **annotation IDs** at this stage. Notebook 03 maps annotation IDs to fracture classes using the combined metadata.


In [ ]:
# Cell 16 — Final preprocessing configuration

TARGET_SPACING = np.array([1.0, 1.0, 1.0], dtype=np.float32)

HU_MIN = -1000.0
HU_MAX = 400.0

CT_DTYPE = np.float32
LABEL_DTYPE = np.int16

PREPROCESSING_CONFIG = {
    "target_spacing_mm": [1.0, 1.0, 1.0],
    "ct_interpolation": "linear",
    "label_interpolation": "nearest_neighbor",
    "hu_clip": [-1000.0, 400.0],
    "ct_normalization": "[0,1]",
    "ct_dtype": "float32",
    "label_dtype": "int16",
    "seed": SEED,
    "cohort_size": 500,
    "split": {
        "train": 350,
        "validation": 75,
        "test": 75,
    },
}

CONFIG_PATH = METADATA_ROOT / "preprocessing_config_500.json"

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(PREPROCESSING_CONFIG, f, indent=2)

print(json.dumps(PREPROCESSING_CONFIG, indent=2))
print("Saved:", CONFIG_PATH)


In [ ]:
# Cell 17 — NIfTI preprocessing functions

def normalize_ct_hu(ct):
    ct = np.asarray(ct, dtype=np.float32)
    ct = np.clip(ct, HU_MIN, HU_MAX)
    ct = (ct - HU_MIN) / (HU_MAX - HU_MIN)
    return ct.astype(CT_DTYPE, copy=False)


def resample_volume(volume, original_spacing, target_spacing, order):
    original_spacing = np.asarray(original_spacing, dtype=np.float32)
    target_spacing = np.asarray(target_spacing, dtype=np.float32)

    zoom_factors = original_spacing / target_spacing

    resampled = zoom(
        volume,
        zoom=zoom_factors,
        order=order,
        mode="nearest",
        prefilter=(order != 0),
    )

    return resampled


def preprocess_nifti_pair(ct_nii, label_nii):
    ct = ct_nii.get_fdata(dtype=np.float32)
    label = label_nii.get_fdata(dtype=np.float32)

    ct_spacing = np.asarray(ct_nii.header.get_zooms()[:3], dtype=np.float32)
    label_spacing = np.asarray(label_nii.header.get_zooms()[:3], dtype=np.float32)

    if not np.allclose(ct_spacing, label_spacing, atol=1e-4):
        raise ValueError(
            f"CT/label spacing mismatch: {ct_spacing} vs {label_spacing}"
        )

    if ct.shape != label.shape:
        raise ValueError(
            f"CT/label shape mismatch: {ct.shape} vs {label.shape}"
        )

    ct = normalize_ct_hu(ct)

    ct_resampled = resample_volume(
        ct,
        ct_spacing,
        TARGET_SPACING,
        order=1,
    ).astype(np.float32)

    label_resampled = resample_volume(
        label,
        label_spacing,
        TARGET_SPACING,
        order=0,
    ).astype(np.int16)

    label_resampled = np.rint(label_resampled).astype(np.int16)

    if ct_resampled.shape != label_resampled.shape:
        raise ValueError(
            f"Resampled CT/label mismatch: "
            f"{ct_resampled.shape} vs {label_resampled.shape}"
        )

    return {
        "ct": ct_resampled,
        "label": label_resampled,
        "original_spacing": ct_spacing.tolist(),
        "target_spacing": TARGET_SPACING.tolist(),
    }


print("Preprocessing functions defined.")


In [ ]:
# Cell 18 — Remote source ZIP index cache

ZIP_INDEX_CACHE = {}

def get_source_zip_index(source_name):
    if source_name in ZIP_INDEX_CACHE:
        return ZIP_INDEX_CACHE[source_name]

    cfg = SOURCES[source_name]

    print(f"Inspecting remote image ZIP: {cfg['image_zip']}")
    size, entries = read_zip64_central_directory(cfg["image_url"])

    ZIP_INDEX_CACHE[source_name] = {
        "size": size,
        "entries": entries,
    }

    print(
        f"Remote ZIP indexed: {len(entries)} members, "
        f"{size / (1024**3):.2f} GB"
    )

    return ZIP_INDEX_CACHE[source_name]


print("ZIP index cache ready.")


In [ ]:
# Cell 19 — Robust patient NIfTI loader

def load_patient_nifti_from_source(patient_id, source_name):
    cfg = SOURCES[source_name]
    index = get_source_zip_index(source_name)
    entries = index["entries"]

    image_member = find_patient_member(entries, patient_id, "image")
    print("Image member:", image_member)
    image_bytes = get_zip_member_bytes(cfg["image_url"], entries[image_member])

    label_zip_path = CACHE_ROOT / cfg["label_zip"]
    cache_small_file(cfg["label_url"], label_zip_path)

    with zipfile.ZipFile(label_zip_path, "r") as zf:
        label_candidates = [
            name for name in zf.namelist()
            if patient_id in Path(name).name
            and "label" in Path(name).name.lower()
        ]
        if not label_candidates:
            raise FileNotFoundError(
                f"No label member found for {patient_id} in {cfg['label_zip']}"
            )
        label_member = sorted(label_candidates)[0]
        label_bytes = zf.read(label_member)

    print("Label member:", label_member)

    # NiBabel is given real temporary .nii.gz paths; BytesIO is not used.
    image_path = None
    label_path = None
    try:
        with tempfile.NamedTemporaryFile(suffix=".nii.gz", delete=False) as f:
            f.write(image_bytes)
            image_path = f.name
        with tempfile.NamedTemporaryFile(suffix=".nii.gz", delete=False) as f:
            f.write(label_bytes)
            label_path = f.name

        ct_nii = nib.load(image_path)
        label_nii = nib.load(label_path)

        # Materialize arrays while temporary files exist.
        ct_data = ct_nii.get_fdata(dtype=np.float32)
        label_data = label_nii.get_fdata(dtype=np.float32)
        ct_affine = ct_nii.affine.copy()
        label_affine = label_nii.affine.copy()
        ct_header = ct_nii.header.copy()
        label_header = label_nii.header.copy()
    finally:
        if image_path and os.path.exists(image_path):
            os.remove(image_path)
        if label_path and os.path.exists(label_path):
            os.remove(label_path)

    ct_nii = nib.Nifti1Image(ct_data, ct_affine, header=ct_header)
    label_nii = nib.Nifti1Image(label_data, label_affine, header=label_header)
    return ct_nii, label_nii

print("Robust patient loader defined.")


In [ ]:
# Cell 20 — Persistent patient preprocessing and save helpers

def patient_image_path(patient_id):
    return IMAGE_ROOT / f"{patient_id}-image.npy"


def patient_label_path(patient_id):
    return LABEL_ROOT / f"{patient_id}-label.npy"


def patient_is_complete(patient_id):
    return (
        patient_image_path(patient_id).exists()
        and patient_label_path(patient_id).exists()
        and patient_image_path(patient_id).stat().st_size > 0
        and patient_label_path(patient_id).stat().st_size > 0
    )


def update_status(status_df, status_path, patient_id, **updates):
    idx = status_df.index[status_df["public_id"] == str(patient_id)]
    if len(idx) != 1:
        raise ValueError(f"Status record missing for {patient_id}")
    for key, value in updates.items():
        if key not in status_df.columns:
            raise KeyError(key)
        status_df.loc[idx[0], key] = "" if value is None else str(value)
    status_df.to_csv(status_path, index=False)


def save_patient_arrays(patient_id, source_name, split_name, ct, label, original_spacing, status_df, status_path):
    ct_path = patient_image_path(patient_id)
    label_path = patient_label_path(patient_id)
    ct_tmp = IMAGE_ROOT / f".{patient_id}-image.tmp.npy"
    label_tmp = LABEL_ROOT / f".{patient_id}-label.tmp.npy"

    np.save(ct_tmp, ct)
    np.save(label_tmp, label)
    os.replace(ct_tmp, ct_path)
    os.replace(label_tmp, label_path)

    update_status(
        status_df, status_path, patient_id,
        status="completed",
        source=source_name,
        split=split_name,
        ct_path=str(ct_path),
        label_path=str(label_path),
        ct_shape=list(ct.shape),
        label_shape=list(label.shape),
        ct_dtype=str(ct.dtype),
        label_dtype=str(label.dtype),
        original_spacing=list(np.asarray(original_spacing, dtype=float)),
        target_spacing=list(TARGET_SPACING.astype(float)),
        ct_min=float(ct.min()),
        ct_max=float(ct.max()),
        label_ids=list(np.unique(label).astype(int)),
        ct_size_mb=ct_path.stat().st_size / (1024**2),
        label_size_mb=label_path.stat().st_size / (1024**2),
        completed_at=time.strftime("%Y-%m-%d %H:%M:%S"),
        error="",
    )
    return ct_path, label_path

print("Persistent save helpers defined.")


## Persistent processing status

`processing_status.csv` is the restart/audit manifest. Every patient has one record. A patient is marked `completed` only after both CT and label files have been written and the status file has been updated.


In [ ]:
# Cell 21 — Create or restore processing-status table

STATUS_PATH = METADATA_ROOT / "processing_status.csv"
STATUS_COLUMNS = [
    "public_id", "source", "split", "status", "ct_path", "label_path",
    "ct_shape", "label_shape", "ct_dtype", "label_dtype", "original_spacing",
    "target_spacing", "ct_min", "ct_max", "label_ids", "ct_size_mb",
    "label_size_mb", "started_at", "completed_at", "error"
]

if STATUS_PATH.exists():
    status_df = pd.read_csv(STATUS_PATH, dtype=str)
else:
    status_df = pd.DataFrame(columns=STATUS_COLUMNS)

for col in STATUS_COLUMNS:
    if col not in status_df.columns:
        status_df[col] = ""
status_df = status_df[STATUS_COLUMNS]

registry_lookup = patient_registry.set_index("public_id")
existing = set(status_df["public_id"].astype(str))

for patient_id in patient_registry["public_id"].astype(str):
    if patient_id not in existing:
        row = registry_lookup.loc[patient_id]
        status_df.loc[len(status_df)] = [
            patient_id, row["source"], row["split"], "pending", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", ""
        ]

status_df.to_csv(STATUS_PATH, index=False)
print("Status records:", len(status_df))
print(status_df["status"].value_counts())
print("Saved:", STATUS_PATH)


In [ ]:
# Cell 22 — Smoke-test configuration

RUN_SMOKE_TEST = True
SMOKE_TEST_PATIENT = "RibFrac128"
SMOKE_TEST_SOURCE = "train_part1"


In [ ]:
# Cell 23 — Run and persist the smoke test
# Cell 23 — Smoke test

RUN_SMOKE_TEST = True
SMOKE_TEST_PATIENT = "RibFrac128"
SMOKE_TEST_SOURCE = "train_part1"

if RUN_SMOKE_TEST:

    ct_nii, label_nii = load_patient_nifti_from_source(
        SMOKE_TEST_PATIENT,
        SMOKE_TEST_SOURCE,
    )

    result = preprocess_nifti_pair(
        ct_nii,
        label_nii,
    )

    # Handle the preprocessing result explicitly.
    ct = result["ct"]
    label = result["label"]
    spacing = result["original_spacing"]

    print("=" * 80)
    print("SMOKE TEST")
    print("=" * 80)

    print("Patient:", SMOKE_TEST_PATIENT)
    print("Original spacing:", spacing)
    print("Target spacing:", TARGET_SPACING)
    print("CT shape:", ct.shape)
    print("Label shape:", label.shape)
    print("CT dtype:", ct.dtype)
    print("Label dtype:", label.dtype)
    print(
        "CT range:",
        float(ct.min()),
        "to",
        float(ct.max())
    )
    print("Label IDs:", np.unique(label))

    assert ct.ndim == 3
    assert label.ndim == 3
    assert ct.shape == label.shape

    assert ct.dtype == np.float32
    assert label.dtype == np.int16

    assert float(ct.min()) >= 0.0
    assert float(ct.max()) <= 1.0

    print("\nSmoke-test preprocessing: PASS")

    del ct_nii
    del label_nii
    del ct
    del label

    gc.collect()

else:
    print("Smoke test disabled.")


## Full 500-patient persistent preprocessing

The next cell is resumable and processes patients one at a time. Set `MAX_PATIENTS_PER_RUN` to a small number while testing. Because a 1-mm CT volume is large, do not assume that all 500 volumes will fit in the Kaggle working disk. If storage is insufficient, process in durable batches and retain the same patient manifest.


In [ ]:
# Cell 24 — Resumable full preprocessing loop

registry_to_process = patient_registry.sort_values(["split", "public_id"]).reset_index(drop=True)
processed_this_run = 0

for _, row in registry_to_process.iterrows():
    patient_id = str(row["public_id"])
    source_name = str(row["source"])
    split_name = str(row["split"])

    current_status = status_df.loc[
        status_df["public_id"] == patient_id, "status"
    ].iloc[0]

    if RESUME_EXISTING and patient_is_complete(patient_id):
        if current_status != "completed":
            update_status(
                status_df, STATUS_PATH, patient_id,
                status="completed", source=source_name, split=split_name,
                ct_path=str(patient_image_path(patient_id)),
                label_path=str(patient_label_path(patient_id)),
            )
        print(f"{patient_id}: already completed — SKIP")
        continue

    if current_status == "failed" and not RETRY_FAILED:
        print(f"{patient_id}: previous failure — SKIP")
        continue

    if MAX_PATIENTS_PER_RUN is not None and processed_this_run >= MAX_PATIENTS_PER_RUN:
        print(f"Reached MAX_PATIENTS_PER_RUN={MAX_PATIENTS_PER_RUN}.")
        break

    print("\n" + "=" * 80)
    print(f"PROCESSING {patient_id} | source={source_name} | split={split_name}")
    print("=" * 80)

    update_status(
        status_df, STATUS_PATH, patient_id,
        status="processing",
        source=source_name,
        split=split_name,
        started_at=time.strftime("%Y-%m-%d %H:%M:%S"),
        error="",
    )

    try:
        ct_nii, label_nii = load_patient_nifti_from_source(patient_id, source_name)
        ct, label, spacing = preprocess_nifti_pair(ct_nii, label_nii)

        save_patient_arrays(
            patient_id, source_name, split_name,
            ct, label, spacing, status_df, STATUS_PATH
        )

        print(f"{patient_id}: COMPLETE | shape={ct.shape}")
        processed_this_run += 1

        del ct_nii, label_nii, ct, label
        gc.collect()

    except Exception as exc:
        error_text = repr(exc)
        print(f"{patient_id}: FAILED → {error_text}")
        update_status(
            status_df, STATUS_PATH, patient_id,
            status="failed", error=error_text, completed_at=""
        )
        gc.collect()

print("Patients completed this run:", processed_this_run)


In [ ]:
# Cell 25 — Persistent output integrity validation

status_df = pd.read_csv(STATUS_PATH, dtype=str)
completed = status_df[status_df["status"] == "completed"].copy()

missing_files = []
for _, row in completed.iterrows():
    patient_id = row["public_id"]
    if not patient_image_path(patient_id).exists():
        missing_files.append((patient_id, "image"))
    if not patient_label_path(patient_id).exists():
        missing_files.append((patient_id, "label"))

print("Completed:", len(completed))
print("Pending:", int((status_df["status"] == "pending").sum()))
print("Failed:", int((status_df["status"] == "failed").sum()))
print("Missing files among completed patients:", len(missing_files))

assert not missing_files, missing_files[:10]
print("Persistent output integrity: PASS")


In [ ]:
# Cell 26 — Validate one persisted patient

completed_ids = completed["public_id"].tolist()
if completed_ids:
    patient_id = completed_ids[0]
    ct = np.load(patient_image_path(patient_id), mmap_mode="r")
    label = np.load(patient_label_path(patient_id), mmap_mode="r")

    print("Patient:", patient_id)
    print("CT shape:", ct.shape)
    print("Label shape:", label.shape)
    print("CT dtype:", ct.dtype)
    print("Label dtype:", label.dtype)
    print("CT range:", float(ct.min()), "to", float(ct.max()))
    print("Label IDs:", np.unique(label))

    assert ct.shape == label.shape
    assert ct.dtype == np.float32
    assert label.dtype == np.int16
    assert float(ct.min()) >= 0.0 and float(ct.max()) <= 1.0
    print("Persisted-volume validation: PASS")
else:
    print("No completed patients yet.")


In [ ]:
# Cell 27 — Final processing report and output inventory

status_df = pd.read_csv(STATUS_PATH, dtype=str)
print("=" * 80)
print("NOTEBOOK 02 — PERSISTENT PROCESSING REPORT")
print("=" * 80)
print("\nStatus:")
print(status_df["status"].value_counts())
print("\nStatus by split:")
print(pd.crosstab(status_df["split"], status_df["status"]))
print("\nStatus by source:")
print(pd.crosstab(status_df["source"], status_df["status"]))

completed = status_df[status_df["status"] == "completed"]
if len(completed):
    print("\nPersisted CT size (GB):", completed["ct_size_mb"].astype(float).sum() / 1024)
    print("Persisted label size (GB):", completed["label_size_mb"].astype(float).sum() / 1024)

print("\nOutput root:", WORK_ROOT)
print("Status file:", STATUS_PATH)
print("\nFiles:")
for path in sorted(WORK_ROOT.rglob("*")):
    if path.is_file():
        size_mb = path.stat().st_size / (1024**2)
        print(f"{path.relative_to(WORK_ROOT)} | {size_mb:.3f} MB")


## Notebook 02 completion criteria

Notebook 02 is complete when all 500 patients have `completed` status and every completed patient has both a CT and label `.npy` file.

The persistent dataset is the input to Notebook 03. Notebook 03 should **not** contact Zenodo or recreate the patient split.

If the Kaggle working disk is insufficient for the complete 500-patient archive, process the registry in durable batches and preserve `patient_source_registry_500.csv`, `patient_split_500.csv`, `processing_status.csv`, and `preprocessing_config.json` as the audit trail.
